# RAG-Based Academic Research Search Engine

In [ ]:
%pip install -q openai faiss-cpu pandas numpy nltk


In [ ]:
import os
import re

import faiss
import numpy as np
import pandas as pd
import nltk

from openai import OpenAI
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY environment variable is not configured.")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
EMBEDDING_MODEL = "text-embedding-3-small"


## Research Paper Dataset

In [ ]:
research_papers = {
    "papers": [
        {
            "title": "Quantum Computing and Modern Cryptography",
            "abstract": "This paper examines the impact of quantum computing on modern cryptographic systems and discusses quantum-resistant cryptographic approaches.",
            "full_text": "Quantum computers use quantum mechanical principles to solve certain computational problems more efficiently. Cryptographic systems based on integer factorization and discrete logarithms are particularly vulnerable to quantum algorithms. The study analyzes Shor algorithm and its implications for public key cryptography while discussing post-quantum cryptography.",
            "keywords": "quantum computing, cryptography, post-quantum cryptography, Shor algorithm",
            "year": 2025
        },
        {
            "title": "Post-Quantum Cryptography: Algorithms and Applications",
            "abstract": "A comprehensive study of cryptographic algorithms designed to remain secure against quantum computer attacks.",
            "full_text": "The emergence of large-scale quantum computers creates significant challenges for traditional public key cryptography. This paper evaluates lattice-based, code-based, hash-based, and multivariate cryptographic algorithms for post-quantum security.",
            "keywords": "post-quantum cryptography, lattice cryptography, quantum security",
            "year": 2024
        },
        {
            "title": "Quantum Key Distribution for Secure Communication",
            "abstract": "This research explores quantum key distribution and its application in secure communication networks.",
            "full_text": "Quantum key distribution provides a mechanism for establishing secret keys using quantum states. The research discusses BB84, quantum measurement, photon polarization, eavesdropping detection, and secure communication.",
            "keywords": "quantum key distribution, QKD, BB84, secure communication",
            "year": 2023
        },
        {
            "title": "Shor Algorithm and Its Impact on Public Key Cryptography",
            "abstract": "This paper analyzes Shor quantum algorithm and its ability to threaten widely used public key cryptographic systems.",
            "full_text": "Shor algorithm provides a polynomial-time quantum solution to integer factorization and discrete logarithm problems. RSA and elliptic curve cryptography rely on the computational difficulty of these problems and may become insecure against sufficiently powerful quantum computers.",
            "keywords": "Shor algorithm, RSA, elliptic curve cryptography, quantum computing",
            "year": 2022
        },
        {
            "title": "Quantum Machine Learning for Scientific Computing",
            "abstract": "An investigation into quantum machine learning techniques and their applications in scientific computation.",
            "full_text": "Quantum machine learning combines quantum computing principles with machine learning algorithms. The paper investigates quantum circuits, variational algorithms, quantum classifiers, and their applications in scientific computing.",
            "keywords": "quantum machine learning, quantum algorithms, scientific computing",
            "year": 2025
        },
        {
            "title": "Advances in Lattice-Based Cryptography",
            "abstract": "This study presents recent developments in lattice-based cryptographic schemes for secure computing.",
            "full_text": "Lattice-based cryptography is considered a promising approach for post-quantum security. The research examines learning with errors, module lattices, structured lattices, encryption mechanisms, and digital signatures.",
            "keywords": "lattice cryptography, post-quantum security, learning with errors",
            "year": 2025
        },
        {
            "title": "Classical Cryptography and Information Security",
            "abstract": "This paper reviews classical cryptographic techniques and their importance in information security.",
            "full_text": "Classical cryptography includes symmetric encryption, asymmetric encryption, hashing, authentication, and digital signatures. The study evaluates common security mechanisms and their applications in computer networks.",
            "keywords": "cryptography, information security, encryption, authentication",
            "year": 2021
        },
        {
            "title": "Quantum Networks and Secure Data Transmission",
            "abstract": "This research investigates quantum networks and secure transmission of information using quantum communication technologies.",
            "full_text": "Quantum networks connect quantum devices through quantum communication channels. The study discusses entanglement distribution, quantum repeaters, quantum communication protocols, and secure information transmission.",
            "keywords": "quantum networks, quantum communication, secure transmission",
            "year": 2024
        },
        {
            "title": "Blockchain Security Using Cryptographic Techniques",
            "abstract": "An analysis of cryptographic mechanisms used to secure blockchain systems.",
            "full_text": "Blockchain systems use cryptographic hashing, digital signatures, public key cryptography, and consensus mechanisms. This paper examines blockchain security and vulnerabilities in cryptographic implementations.",
            "keywords": "blockchain, cryptography, digital signatures, security",
            "year": 2023
        },
        {
            "title": "Quantum Threats to Internet Security",
            "abstract": "This paper examines potential quantum computing threats to current Internet security infrastructure.",
            "full_text": "Large-scale quantum computers could compromise cryptographic protocols currently protecting Internet communications. The research examines quantum attacks against RSA and elliptic curve systems and discusses migration strategies toward quantum-resistant algorithms.",
            "keywords": "quantum threats, Internet security, RSA, post-quantum security",
            "year": 2025
        }
    ]
}

papers_df = pd.DataFrame(research_papers["papers"])
papers_df


In [ ]:
required_fields = {"title", "abstract", "full_text", "keywords", "year"}
missing_fields = required_fields.difference(papers_df.columns)

if missing_fields:
    raise ValueError(f"Missing required fields: {sorted(missing_fields)}")

if papers_df.empty:
    raise ValueError("Research-paper dataset cannot be empty.")

for field in required_fields:
    if papers_df[field].isnull().any():
        raise ValueError(f"Field '{field}' contains missing values.")


## Preprocessing and Tokenization

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def preprocess_text(text):
    text = clean_text(text).lower()
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if re.search(r"[a-z0-9]", token)]
    return " ".join(tokens)


text_columns = ["title", "abstract", "full_text", "keywords"]

for column in text_columns:
    papers_df[column] = papers_df[column].map(clean_text)
    papers_df[f"processed_{column}"] = papers_df[column].map(preprocess_text)

papers_df["year"] = pd.to_numeric(papers_df["year"], errors="coerce")

if papers_df["year"].isnull().any():
    raise ValueError("Every research paper must have a valid publication year.")

papers_df["year"] = papers_df["year"].astype(int)

papers_df[["title", "year", "processed_title"]].head()


## Create Documents for Embeddings

In [ ]:
def create_embedding_document(row):
    fields = [
        ("Title", row["processed_title"]),
        ("Abstract", row["processed_abstract"]),
        ("Keywords", row["processed_keywords"]),
        ("Content", row["processed_full_text"])
    ]

    return "\n".join(
        f"{name}: {value}"
        for name, value in fields
        if value
    )


papers_df["embedding_document"] = papers_df.apply(
    create_embedding_document,
    axis=1
)

documents = papers_df["embedding_document"].tolist()

if any(not document.strip() for document in documents):
    raise ValueError("Embedding documents cannot be empty.")

print(f"Documents created: {len(documents)}")


## Generate Embeddings

In [ ]:
def generate_embeddings(texts, batch_size=100):
    if not texts:
        raise ValueError("No documents were provided for embedding.")

    if not isinstance(batch_size, int) or batch_size <= 0:
        raise ValueError("batch_size must be a positive integer.")

    embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch
        )

        embeddings.extend(
            item.embedding for item in response.data
        )

    result = np.asarray(embeddings, dtype="float32")

    if len(result) != len(texts):
        raise ValueError("Embedding count does not match document count.")

    return result


document_embeddings = generate_embeddings(documents)

print(f"Embedding shape: {document_embeddings.shape}")


## Store Embeddings in FAISS

In [ ]:
def build_faiss_index(embeddings):
    if embeddings.ndim != 2:
        raise ValueError("Embeddings must be a two-dimensional array.")

    if embeddings.shape[0] == 0:
        raise ValueError("Cannot create FAISS index from empty embeddings.")

    vectors = embeddings.copy()
    faiss.normalize_L2(vectors)

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)

    return index


index = build_faiss_index(document_embeddings)

if index.ntotal != len(papers_df):
    raise ValueError("FAISS index size does not match dataset size.")

metadata = papers_df[
    ["title", "abstract", "full_text", "keywords", "year"]
].to_dict(orient="records")

if len(metadata) != index.ntotal:
    raise ValueError("Metadata count does not match FAISS index size.")

print(f"FAISS vectors stored: {index.ntotal}")


## Query Embedding

In [ ]:
def embed_query(query):
    query = clean_text(query)

    if not query:
        raise ValueError("Query cannot be empty.")

    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[query]
    )

    query_embedding = np.asarray(
        [response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_embedding)

    return query_embedding


## Semantic Retrieval

In [ ]:
def semantic_search(query, candidate_count=None):
    if index.ntotal == 0:
        return []

    query_embedding = embed_query(query)

    if candidate_count is None:
        candidate_count = index.ntotal

    if not isinstance(candidate_count, int) or candidate_count <= 0:
        raise ValueError("candidate_count must be a positive integer.")

    candidate_count = min(candidate_count, index.ntotal)

    similarities, indices = index.search(
        query_embedding,
        candidate_count
    )

    results = []

    for similarity, idx in zip(similarities[0], indices[0]):
        if idx < 0:
            continue

        result = metadata[int(idx)].copy()
        result["semantic_similarity"] = float(similarity)
        results.append(result)

    return results


## Relevance and Recency Ranking

In [ ]:
def rank_by_relevance_and_recency(results, top_k):
    if not isinstance(top_k, int):
        raise ValueError("top_k must be an integer.")

    if top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    if len(results) < top_k:
        raise ValueError(
            "The dataset does not contain enough documents for the requested top_k."
        )

    result_df = pd.DataFrame(results)
    result_df["year"] = pd.to_numeric(
        result_df["year"],
        errors="coerce"
    )

    if result_df["year"].isnull().any():
        raise ValueError(
            "Retrieved documents contain an invalid publication year."
        )

    max_year = result_df["year"].max()
    min_year = result_df["year"].min()

    if max_year == min_year:
        result_df["recency_score"] = 1.0
    else:
        result_df["recency_score"] = (
            result_df["year"] - min_year
        ) / (max_year - min_year)

    result_df["combined_score"] = (
        0.75 * result_df["semantic_similarity"]
        + 0.25 * result_df["recency_score"]
    )

    result_df = result_df.sort_values(
        by=["combined_score", "semantic_similarity", "year"],
        ascending=[False, False, False]
    ).head(top_k)

    result_df = result_df.sort_values(
        by=["year", "combined_score", "semantic_similarity"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    return result_df.to_dict(orient="records")


## Main Retrieval Function

In [ ]:
def retrieve_papers(query, top_k=5):
    if not isinstance(top_k, int):
        raise ValueError("top_k must be an integer.")

    if top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    if top_k > index.ntotal:
        raise ValueError(
            f"top_k cannot exceed the number of available documents: {index.ntotal}."
        )

    semantic_results = semantic_search(
        query,
        candidate_count=index.ntotal
    )

    return rank_by_relevance_and_recency(
        semantic_results,
        top_k
    )


## Search

In [ ]:
def search(query, top_k=5):
    results = retrieve_papers(query, top_k)

    if len(results) != top_k:
        raise ValueError(
            "The search engine could not return the requested number of documents."
        )

    return results


## Required Scenario

In [ ]:
query = "Quantum Computing in Cryptography"
top_k = 5

results = search(
    query,
    top_k
)

results_df = pd.DataFrame(results)

results_df[
    [
        "title",
        "year",
        "semantic_similarity",
        "recency_score",
        "combined_score"
    ]
]


## Display Retrieved Papers

In [ ]:
def display_results(results):
    if not results:
        print("No relevant research papers found.")
        return

    for rank, paper in enumerate(results, start=1):
        print(f"Rank: {rank}")
        print(f"Title: {paper['title']}")
        print(f"Year: {paper['year']}")
        print(
            f"Semantic Similarity: "
            f"{paper['semantic_similarity']:.4f}"
        )
        print(
            f"Recency Score: "
            f"{paper['recency_score']:.4f}"
        )
        print(
            f"Combined Score: "
            f"{paper['combined_score']:.4f}"
        )
        print(f"Abstract: {paper['abstract']}")
        print("-" * 80)


display_results(results)


## Additional Search Test

In [ ]:
test_query = "post quantum cryptography and RSA security"
test_results = search(test_query, top_k=3)
display_results(test_results)
